# KV Cache from Scratch

A KV cache added to the character-level GPT from Andrej Karpathy's [Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY) lecture ([original code](https://github.com/karpathy/ng-video-lecture)): 10.8M parameters, 6 layers × 6 heads, Pre-LN, Tiny Shakespeare.

**What changed vs. the original model** (each change is marked `KV cache:` in the model cell):
1. `Head`: append the new K/V to a per-head cache, and take the **last T rows** of the causal mask
2. `MultiHeadAttention` / `Block`: pass each head its own cache
3. `GPTLanguageModel.forward`: positions start at `start_pos`; loop over the blocks instead of `nn.Sequential`
4. `generate_cached`: **prefill** the prompt once, then **decode** one token per step

**Run it:** Runtime → Change runtime type → T4 GPU, then run all cells. Training is optional (`TRAIN = False`): the correctness test and the benchmark don't need trained weights.

Results are summarized in Section 7.

## 1. Setup

In [1]:
!nvidia-smi -L
!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!ls -lh input.txt

GPU 0: Tesla T4 (UUID: GPU-5482509f-1248-3ce7-acb6-84ed23b69f54)
-rw-r--r-- 1 root root 1.1M Nov 17 07:58 input.txt


In [2]:
import os, time
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters (same as the video)
batch_size = 64
block_size = 256      # max context length; the KV cache can't go past this (learned pos-emb)
max_iters = 5000      # lower this (e.g. 2000) if training is too slow on a T4
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

# notebook switches
TRAIN = False                                       # True = train; False = skip (fine for test + benchmark)
WEIGHTS = "/content/gpt_shakespeare.pt"             # where trained weights are saved/loaded
SAVE_TO_DRIVE = False                               # True = also copy weights to Google Drive

torch.manual_seed(1337)
print("device:", device)

device: cuda


## 2. Data

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("vocab_size:", vocab_size, "| tokens:", len(data))

vocab_size: 65 | tokens: 1115394


## 3. Model with a KV cache

The cache is a list (one per layer) of lists (one per head) of dicts: `cache[layer][head] = {'k': (B, Tk, hs), 'v': (B, Tk, hs)}`, where `Tk` is the number of tokens seen so far.
With `cache=None` the model behaves exactly like the original, so training is unchanged.

In [ ]:
# Karpathy's GPT (ng-video-lecture/gpt.py) with a KV cache added.
# Every change vs. the original is marked "KV cache:".

class Head(nn.Module):
    # one head of self-attention

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q = self.query(x)   # (B, T, hs)
        k = self.key(x)     # (B, T, hs)
        v = self.value(x)   # (B, T, hs)

        # KV cache: put the past K/V in front of the new ones, then store the longer k, v.
        # `cache` is this head's dict (None during training).
        if cache is not None:
            if 'k' in cache:
                k = torch.cat([cache['k'], k], dim=1)   # (B, Tk, hs)
                v = torch.cat([cache['v'], v], dim=1)
            cache['k'], cache['v'] = k, v
        Tk = k.shape[1]                                  # all tokens so far
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5     # (B, T, Tk)

        # KV cache: the T queries are the LAST T of the Tk positions, so use those rows of the mask.
        wei = wei.masked_fill(self.tril[Tk-T:Tk, :Tk] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v      # (B, T, hs)


class MultiHeadAttention(nn.Module):
    # multiple heads of self-attention in parallel

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, caches=None):
        # KV cache: head i gets its own cache dict, caches[i]
        out = torch.cat([h(x, None if caches is None else caches[i])
                         for i, h in enumerate(self.heads)], dim=-1)   # (B, T, C)
        return self.dropout(self.proj(out))


class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)  # same shape


class Block(nn.Module):
    # Transformer block: communication followed by computation (pre-LN)

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, caches=None):
        x = x + self.sa(self.ln1(x), caches)   # KV cache: pass the caches through
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, cache=None, start_pos=0):
        # cache: list (one per layer) of lists (one per head) of dicts, or None
        # start_pos: absolute position of idx[:, 0] (0 for prefill / training)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)   # (B, T, C)
        # KV cache: positions start at start_pos (= tokens already in the cache), not 0
        pos_emb = self.position_embedding_table(torch.arange(start_pos, start_pos + T, device=device))
        x = tok_emb + pos_emb   # (B, T, C)
        # KV cache: nn.Sequential can't pass extra arguments, so loop over the blocks
        for i, block in enumerate(self.blocks):
            x = block(x, None if cache is None else cache[i])
        x = self.ln_f(x)
        logits = self.lm_head(x)   # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # Original (no cache): re-runs the WHOLE sequence for every new token
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]   # (B, T)
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    @torch.no_grad()
    def generate_cached(self, idx, max_new_tokens):
        # KV cache: prefill the prompt once, then decode one token per step
        cache = new_cache()
        logits, _ = self(idx, cache=cache, start_pos=0)   # PREFILL: stores the prompt's K/V
        pos = idx.shape[1]                                 # position of the next token
        for i in range(max_new_tokens):
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)   # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
            if i == max_new_tokens - 1 or pos >= block_size:
                break                                      # done, or out of learned positions
            logits, _ = self(idx_next, cache=cache, start_pos=pos)   # DECODE: only the new token
            pos += 1
        return idx


def new_cache():
    # empty KV cache: cache[layer][head] = {}
    return [[{} for _ in range(n_head)] for _ in range(n_layer)]

In [5]:
# (re)build the model; run this cell again every time you edit the model cell
model = GPTLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

if os.path.exists(WEIGHTS):
    model.load_state_dict(torch.load(WEIGHTS, map_location=device))
    print("loaded trained weights from", WEIGHTS)

10.788929 M parameters


## 4. (Optional) Train

Only needed if you want readable Shakespeare. Set `TRAIN = True` in Section 1. On a T4, 5000 iterations takes a while. Lower `max_iters` if you're impatient. The weights are saved, so you only train once per session (or once ever, if you use Drive).

In [6]:
if TRAIN:
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    t0 = time.time()
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss()
            print(f"step {it}: train {losses['train']:.4f}, val {losses['val']:.4f}  ({time.time()-t0:.0f}s)")
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    torch.save(model.state_dict(), WEIGHTS)
    print("saved to", WEIGHTS)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        !cp {WEIGHTS} /content/drive/MyDrive/
        print("copied to Google Drive")
else:
    print("TRAIN = False -> skipping training (random weights are fine for the test + benchmark)")

TRAIN = False -> skipping training (random weights are fine for the test + benchmark)


In [7]:
# baseline sample with the ORIGINAL generate()
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


?qf;xBDkRZkNdc.wj,ZTkOLrL,xHtK
bHxPWokMBbzA$3:.aSvgO-3DSMBF?gLTa
hXbYVXJthXMNuDqh$kxv.tbVr dXlrDvaLeWFw3$HPDAWk,fDE'aYzxzkkouX
Yo3&$LMtofViHIvB!H&V!$W;KdYlNx,
e3 irYeYEvnkciK;lxW;HFGEdroG EsSXUB;xWk J..GD3.FeWjbm!selJljnFAUVQu.$-huD3qoVcrvVN:?UuO;Mnis'X3Uwty.HJ'vBPUHIkBBf&pjY-igvIEjVk:D.yqwJdqNBtSkk


## 5. Correctness test ✅

The cached path must give **the same logits** as a full recompute: prefill the first 49 tokens, decode the 50th, and compare with one forward pass over all 50.

If it fails, the usual causes are the mask slice in `Head.forward`, the `start_pos` offset in `GPTLanguageModel.forward`, or dropout still being on (the cell calls `model.eval()`).

In [8]:
model.eval()   # IMPORTANT: dropout off, otherwise outputs are random
with torch.no_grad():
    L = 50
    seq = torch.randint(0, vocab_size, (1, L), device=device)

    full_logits, _ = model(seq)                                  # no cache, all L tokens

    cache = new_cache()
    pre_logits, _ = model(seq[:, :-1], cache=cache, start_pos=0)          # prefill L-1 tokens
    step_logits, _ = model(seq[:, -1:], cache=cache, start_pos=L - 1)     # decode the last one

print("cache filled:", 'k' in cache[0][0], "| layer-0 head-0 k shape:", cache[0][0].get('k', torch.empty(0)).shape)
print("prefill matches:", torch.allclose(full_logits[:, :-1], pre_logits, atol=1e-4))
print("decode  matches:", torch.allclose(full_logits[:, -1], step_logits[:, -1], atol=1e-4))
print("max abs diff   :", (full_logits[:, -1] - step_logits[:, -1]).abs().max().item())

cache filled: True | layer-0 head-0 k shape: torch.Size([1, 50, 64])
prefill matches: True
decode  matches: True
max abs diff   : 5.289912223815918e-07


In [ ]:
# End-to-end: with GREEDY decoding both generators must produce the same tokens.
# This check caught a bug the logit test above can't see: the prefill ran with cache=None.
# (we temporarily swap sampling for argmax so the randomness doesn't hide bugs)
_orig_multinomial = torch.multinomial
torch.multinomial = lambda probs, num_samples: probs.argmax(dim=-1, keepdim=True)
try:
    ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
    a = model.generate(ctx, 100)
    b = model.generate_cached(ctx, 100)
    print("same tokens:", torch.equal(a, b))
finally:
    torch.multinomial = _orig_multinomial

## 6. Benchmark ⏱️

Tokens/sec with and without the cache, for generation lengths 25–250 (they must stay below `block_size`).
Timing: one warm-up call, `torch.cuda.synchronize()` around each measurement (GPU work is asynchronous), best of 3 runs.

In [ ]:
def bench(fn, n_tokens, repeats=3):
    idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    best = float('inf')
    for _ in range(repeats):
        if device == 'cuda': torch.cuda.synchronize()
        t = time.time()
        fn(idx, n_tokens)
        if device == 'cuda': torch.cuda.synchronize()
        best = min(best, time.time() - t)
    return n_tokens / best   # tokens / sec

model.eval()
# warm-up (CUDA init, kernel caching) - don't time this
model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 10)
model.generate_cached(torch.zeros((1, 1), dtype=torch.long, device=device), 10)

lengths = [25, 50, 100, 150, 200, 250]      # must stay < block_size
results = []
for n_tok in lengths:
    a = bench(model.generate, n_tok)
    b = bench(model.generate_cached, n_tok)
    results.append((n_tok, a, b))
    print(f"n={n_tok:4d} | no cache {a:7.1f} tok/s | kv cache {b:7.1f} tok/s | speedup {b/a:4.2f}x")

In [ ]:
import matplotlib.pyplot as plt

ns, nc, kv = zip(*results)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ns, nc, marker='o', label='no cache')
ax[0].plot(ns, kv, marker='o', label='KV cache')
ax[0].set_xlabel('tokens generated'); ax[0].set_ylabel('tokens / sec'); ax[0].set_title('Throughput'); ax[0].legend()
ax[1].plot(ns, [b / a for a, b in zip(nc, kv)], marker='o', color='green')
ax[1].set_xlabel('tokens generated'); ax[1].set_ylabel('speedup (x)'); ax[1].set_title('KV cache speedup')
plt.tight_layout(); plt.savefig('kv_cache_benchmark.png', dpi=150); plt.show()

## 7. Results

| | Result |
|---|---|
| Correctness | Prefill and decode logits match the full recompute: max abs diff **5.3e-7** (FP32 rounding) |
| Throughput (T4, lengths 25–150) | No cache ≈ **59 tok/s** at every length; KV cache **0.68–0.96×** of that, i.e. no speedup |

**Why no speedup?** The no-cache path recomputes the whole sequence, so its work grows with length, yet its per-token latency stays at ≈ 17 ms. The bottleneck is not compute: each token launches hundreds of tiny kernels (one set per head per layer), and CPU-side launch/dispatch overhead dominates. The cache even adds 72 `torch.cat` calls per step (K and V × 6 heads × 6 layers), so it is slightly slower. A KV cache saves FLOPs; it only pays off when FLOPs are the bottleneck (bigger models, longer contexts, CPU).

> Note: the throughput numbers above are from the first run, before fixing a bug in `generate_cached` (the prefill ran with `cache=None`). The fix doesn't change the per-step work; re-run Sections 5–6 to refresh the outputs.

**Next:** fuse Q/K/V for all heads into one matmul (far fewer kernels), try CUDA Graphs / `torch.compile` to remove launch overhead, and repeat on CPU or a bigger model where compute dominates.